# Phần A: Đọc dữ liệu

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Khởi tạo Spark
spark = SparkSession.builder.appName("IoTSensorAnalysis_Large").getOrCreate()

# --- Phần A. Đọc dữ liệu ---
df_iot = spark.read.csv("../data/iot_sensors.csv", header=True, inferSchema=True)

print("Schema dữ liệu cảm biến IoT:")
df_iot.printSchema()
print(f"Tổng số dòng dữ liệu: {df_iot.count()}")




Schema dữ liệu cảm biến IoT:
root
 |-- sensor_id: string (nullable = true)
 |-- temperature: double (nullable = true)
 |-- humidity: double (nullable = true)
 |-- vibration: double (nullable = true)
 |-- status: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)

Tổng số dòng dữ liệu: 1000000


# Phần B: Phân tích thống kê

In [2]:

# 1. Thống kê nhiệt độ toàn hệ thống
print("\nThống kê nhiệt độ (Toàn nhà máy):")
df_iot.select(
    F.avg("temperature").alias("Nhiệt độ TB"),
    F.max("temperature").alias("Nhiệt độ Max"),
    F.min("temperature").alias("Nhiệt độ Min")
).show()

# 2. Độ rung trung bình theo từng cảm biến
print("Độ rung trung bình theo từng cảm biến (Top 10):")
df_iot.groupBy("sensor_id") \
    .agg(F.round(F.avg("vibration"), 2).alias("vibration_avg")) \
    .orderBy(F.desc("vibration_avg")) \
    .show(10)





Thống kê nhiệt độ (Toàn nhà máy):
+-----------------+------------+------------+
|      Nhiệt độ TB|Nhiệt độ Max|Nhiệt độ Min|
+-----------------+------------+------------+
|40.00085560000017|       107.5|       -30.7|
+-----------------+------------+------------+

Độ rung trung bình theo từng cảm biến (Top 10):
+---------+-------------+
|sensor_id|vibration_avg|
+---------+-------------+
|     S004|         15.4|
|     S033|        15.36|
|     S003|        15.31|
|     S059|         15.3|
|     S023|        15.28|
|     S053|        15.26|
|     S074|        15.24|
|     S058|        15.23|
|     S094|        15.22|
|     S072|        15.21|
+---------+-------------+
only showing top 10 rows



# Phần C: Phát hiện bất thường

In [3]:

# 1. Lọc các dòng có nhiệt độ > 80 hoặc độ rung > 50
anomalies = df_iot.filter((F.col("temperature") > 80) | (F.col("vibration") > 50))

print(f"\nSố lượng bản ghi bất thường phát hiện được: {anomalies.count()}")

# 2. Đếm số lượng bất thường theo sensor_id
print("Top 10 cảm biến có nhiều cảnh báo nhất:")
anomalies.groupBy("sensor_id").count().orderBy(F.desc("count")).show(10)





Số lượng bản ghi bất thường phát hiện được: 39476
Top 10 cảm biến có nhiều cảnh báo nhất:
+---------+-----+
|sensor_id|count|
+---------+-----+
|     S074|  442|
|     S020|  439|
|     S072|  434|
|     S057|  434|
|     S059|  434|
|     S017|  426|
|     S013|  424|
|     S055|  423|
|     S022|  422|
|     S004|  421|
+---------+-----+
only showing top 10 rows



# Phần D: Phân tích thời gian

In [4]:

# 1. Tạo cột ngày và tính nhiệt độ TB theo ngày
df_iot = df_iot.withColumn("date", F.to_date(F.col("timestamp")))

print("\nNhiệt độ trung bình theo ngày (Xu hướng):")
df_iot.groupBy("date") \
    .agg(F.round(F.avg("temperature"), 2).alias("daily_avg_temp")) \
    .orderBy("date") \
    .show()

# 2. Tính số lượng trạng thái status theo từng loại
print("Thống kê trạng thái vận hành:")
df_iot.groupBy("status").count().orderBy(F.desc("count")).show()


Nhiệt độ trung bình theo ngày (Xu hướng):
+----------+--------------+
|      date|daily_avg_temp|
+----------+--------------+
|2026-04-01|         39.94|
|2026-04-02|         39.97|
|2026-04-03|         39.93|
|2026-04-04|         40.16|
|2026-04-05|          40.0|
|2026-04-06|         39.96|
|2026-04-07|         39.97|
|2026-04-08|         40.03|
|2026-04-09|         40.06|
|2026-04-10|         39.87|
|2026-04-11|         40.01|
|2026-04-12|          40.0|
|2026-04-13|          39.9|
|2026-04-14|         39.96|
|2026-04-15|         39.99|
|2026-04-16|         39.91|
|2026-04-17|         40.01|
|2026-04-18|         40.03|
|2026-04-19|         39.97|
|2026-04-20|          40.1|
+----------+--------------+
only showing top 20 rows

Thống kê trạng thái vận hành:
+--------+------+
|  status| count|
+--------+------+
|  normal|900241|
| warning| 69800|
|critical| 29959|
+--------+------+



# Thảo luận
1. Dữ liệu IoT thể hiện rõ nhất ở đặc trung Velocity của Big Data:
- Velocity (Tốc độ): Đây là đặc trưng nổi bật nhất vì cảm biến gửi dữ liệu liên tục theo thời gian thực (tần suất có thể tính bằng mili giây). Ngoài ra còn có Volume (số lượng cảm biến lớn tạo ra kho dữ liệu khổng lồ).

2. Dữ liệu cảm biến cần xử lý gần thời gian thực
- An toàn và Bảo trì dự báo: Để phát hiện ngay lập tức các sự cố (cháy nổ, hỏng máy) và ngăn chặn thiệt hại trước khi nó xảy ra. Chờ xử lý theo lô (batch) có thể là quá muộn đối với các thiết bị công nghiệp.